# Notebook 3: Profitability Analysis

Objective: Directly answer: 'Which branches are making or losing money—and why?'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Load cleaned data
df = pd.read_csv('../data/processed/cleaned_restaurant_data.csv')
df['date'] = pd.to_datetime(df['date'])
df['margin'] = df['profit'] / df['revenue']
df.head()

In [ ]:
# Create KPIs
branch_kpis = df.groupby('branch_name').agg(
    total_revenue=('revenue', 'sum'),
    total_costs=('costs', 'sum'),
    net_profit=('profit', 'sum'),
    profit_margin=('margin', 'mean'),
    cost_ratio=('costs', lambda x: x.sum() / df.loc[x.index, 'revenue'].sum()),
    revenue_per_customer=('revenue', 'sum') / df.groupby('branch_name')['customers'].sum()
).reset_index()

branch_kpis['profit_margin'] *= 100  # to %
branch_kpis['cost_ratio'] *= 100
print(branch_kpis.head())

In [ ]:
# Branch segmentation
def classify_profit(profit):
    if profit > 0:
        if profit > branch_kpis['net_profit'].quantile(0.75):
            return 'High Profit'
        elif profit > branch_kpis['net_profit'].quantile(0.5):
            return 'Moderate'
        else:
            return 'Low Profit'
    else:
        return 'Loss-Making'

branch_kpis['tier'] = branch_kpis['net_profit'].apply(classify_profit)

# 9. Branch segmentation
plt.figure(figsize=(10,6))
sns.countplot(data=branch_kpis, x='tier', order=['High Profit', 'Moderate', 'Low Profit', 'Loss-Making'])
plt.title('Branch Segmentation by Profit Tiers')
plt.ylabel('Number of Branches')
plt.savefig('../reports/09_branch_segmentation.png')
plt.show()

In [ ]:
# 10. Loss-making branches
loss_makers = branch_kpis[branch_kpis['tier'] == 'Loss-Making']
plt.figure(figsize=(10,6))
sns.barplot(data=loss_makers, x='net_profit', y='branch_name')
plt.title('Loss-Making Branches')
plt.xlabel('Net Profit (KES)')
plt.savefig('../reports/10_loss_making_branches.png')
plt.show()

In [ ]:
# 11. Profit drivers (correlation)
corr = df[['revenue', 'customers', 'avg_order_value', 'costs', 'profit']].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Profit Drivers Correlation')
plt.savefig('../reports/11_profit_drivers.png')
plt.show()

In [ ]:
# Root cause analysis for loss-makers
for _, row in loss_makers.iterrows():
    branch = row['branch_name']
    branch_data = df[df['branch_name'] == branch]
    avg_customers = branch_data['customers'].mean()
    avg_aov = branch_data['avg_order_value'].mean()
    avg_food_cost = branch_data['costs_food'].mean() / branch_data['revenue'].mean()
    avg_staff_cost = branch_data['costs_staff'].mean() / branch_data['revenue'].mean()
    avg_rent = branch_data['costs_rent'].mean() / branch_data['revenue'].mean()
    
    issues = []
    if avg_customers < df['customers'].quantile(0.25):
        issues.append('low customer volume')
    if avg_aov < df['avg_order_value'].quantile(0.25):
        issues.append('low pricing (AOV)')
    if avg_food_cost > df['costs_food'].quantile(0.75) / df['revenue'].quantile(0.75):
        issues.append('high food cost')
    if avg_staff_cost > df['costs_staff'].quantile(0.75) / df['revenue'].quantile(0.75):
        issues.append('high staffing cost')
    if avg_rent > df['costs_rent'].quantile(0.75) / df['revenue'].quantile(0.75):
        issues.append('rent pressure')
    
    main_issue = '; '.join(issues) if issues else 'multiple factors'
    print(f"{branch}: {main_issue}")
    
    # Update branch_kpis
    branch_kpis.loc[branch_kpis['branch_name'] == branch, 'main_issue'] = main_issue

In [ ]:
# Output dataset
branch_kpis[['branch_name', 'total_revenue', 'net_profit', 'profit_margin', 'main_issue']].to_csv('../reports/branch_profitability_summary.csv', index=False)
print('Branch profitability summary saved.')